# Task3 教学流程：双均线策略与回测

## Goal

本 Notebook 演示如何从本地前复权行情出发，计算 MA5/MA15、识别金叉与死叉、将信号转换为下一交易期仓位，并计算累计回报、最大回撤和夏普比率。

## Setup

### Key Assumptions

- 初始资金 100,000 元，只做多，不加杠杆。
- 当日收盘后形成信号，下一交易期才应用仓位。
- 单边综合交易成本设为 0.1%，无风险利率设为 0。
- 使用前复权日线价格，避免除权除息造成机械跳空。

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "Task3").exists():
    ROOT = ROOT.parent
SCRIPTS = ROOT / "Task3" / "scripts"
sys.path.insert(0, str(SCRIPTS))

from config import CROSS_TOLERANCE, DEFAULT_TRANSACTION_COST, PARAMETER_PAIRS, UNIVERSE, parameter_label
from strategy_backtest import load_prices, run_backtest, summarize_backtest

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

## Steps

### 1. 加载本地行情数据

In [2]:
stock = UNIVERSE[0]  # 平安银行
prices = load_prices(stock)
print(stock)
print(f"数据区间：{prices['trade_date'].min().date()} 至 {prices['trade_date'].max().date()}，共 {len(prices)} 个交易日")
prices[["trade_date", "open", "high", "low", "close", "vol"]].head()

StockSpec(ts_code='000001.SZ', stock_name='平安银行', industry='银行')
数据区间：2019-01-02 至 2026-07-10，共 1823 个交易日


,trade_date,open,high,low,close,vol
0,2019-01-02,7.2975,7.3208,7.1188,7.1421,539386.3200
1,2019-01-03,7.1343,7.2509,7.1110,7.2120,415537.9500
2,2019-01-04,7.1809,7.6317,7.1654,7.5773,1481159.0600
3,2019-01-07,7.6472,7.6550,7.4840,7.5695,865687.6600
4,2019-01-08,7.5617,7.5695,7.4762,7.5073,402388.1100


### 2. 计算 MA5、MA15 和交易信号

In [3]:
result = run_backtest(prices, short_window=5, long_window=15)
signal_rows = result.loc[result["execution"].ne("HOLD"), [
    "trade_date", "close", "short_ma", "long_ma", "signal", "execution", "position"
]]
signal_rows.head(10)

,trade_date,close,short_ma,long_ma,signal,execution,position
15,2019-01-23,8.0436,8.0280,7.8311,HOLD,BUY,1
46,2019-03-14,9.6601,9.6025,9.7124,HOLD,SELL,0
51,2019-03-21,9.8621,9.8917,9.8466,HOLD,BUY,1
54,2019-03-26,9.4036,9.6740,9.7424,HOLD,SELL,0
60,2019-04-03,10.4450,10.1061,9.8580,HOLD,BUY,1
80,2019-05-07,10.0642,10.5009,10.8522,HOLD,SELL,0
106,2019-06-13,9.7844,9.6476,9.5378,HOLD,BUY,1
127,2019-07-12,11.0890,10.7435,10.7920,HOLD,SELL,0
132,2019-07-19,10.9870,10.8534,10.8655,SELL,BUY,1
133,2019-07-22,10.8770,10.8299,10.8613,HOLD,SELL,0


`signal` 是收盘后观察到的金叉或死叉；`execution` 是错后一日实际应用的仓位变化。两列分开可以清楚证明回测没有把信号日已经发生的收益算进去。

### 3. 绘制价格、均线和买卖执行点

In [4]:
buys = result["execution"].eq("BUY")
sells = result["execution"].eq("SELL")

fig, axis = plt.subplots(figsize=(13, 5))
axis.plot(result["trade_date"], result["close"], label="前复权收盘价", color="#1f2937", linewidth=1.2)
axis.plot(result["trade_date"], result["short_ma"], label="MA5", color="#2563eb")
axis.plot(result["trade_date"], result["long_ma"], label="MA15", color="#d97706")
axis.scatter(result.loc[buys, "trade_date"], result.loc[buys, "close"], marker="^", color="#0f766e", label="买入执行")
axis.scatter(result.loc[sells, "trade_date"], result.loc[sells, "close"], marker="v", color="#be123c", label="卖出执行")
axis.set_title(f"{stock.stock_name} MA5/MA15 双均线信号")
axis.set_ylabel("价格（元）")
axis.grid(alpha=0.3)
axis.legend(ncol=5)
plt.show()

C:\Users\13377\AppData\Local\Temp\ipykernel_50492\1647711805.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. 计算策略绩效

In [5]:
metrics = summarize_backtest(result)
pd.Series(metrics, name="MA5/MA15")

observations            1823.0000
cumulative_return         -0.2487
annualized_return         -0.0388
annualized_volatility      0.2113
sharpe_ratio              -0.0826
max_drawdown              -0.5770
benchmark_return           0.4632
excess_return             -0.7118
buy_count                 80.0000
sell_count                79.0000
holding_ratio              0.4778
total_cost_rate            0.1590
Name: MA5/MA15, dtype: float64

### 5. 比较不同均线周期

In [6]:
parameter_rows = []
for short_window, long_window in PARAMETER_PAIRS:
    parameter_result = run_backtest(prices, short_window, long_window)
    parameter_metrics = summarize_backtest(parameter_result, start_date="2024-01-01")
    parameter_rows.append({
        "参数": parameter_label(short_window, long_window),
        "累计回报": parameter_metrics["cumulative_return"],
        "最大回撤": parameter_metrics["max_drawdown"],
        "夏普比率": parameter_metrics["sharpe_ratio"],
        "买入次数": parameter_metrics["buy_count"],
    })

parameter_comparison = pd.DataFrame(parameter_rows).sort_values("夏普比率", ascending=False)
parameter_comparison

,参数,累计回报,最大回撤,夏普比率,买入次数
3,MA10/MA30,-0.0096,-0.1864,0.0594,13
0,MA5/MA10,-0.0259,-0.1900,0.0217,35
2,MA10/MA20,-0.0715,-0.2215,-0.0997,18
1,MA5/MA15,-0.0846,-0.2170,-0.1315,31
4,MA20/MA60,-0.1094,-0.2743,-0.2004,9


## Checks

下面用简单断言检查交易规则、信号错位和绩效指标范围。

In [7]:
expected_target = ((result["short_ma"] > result["long_ma"] + CROSS_TOLERANCE) & result["long_ma"].notna()).astype(int)
expected_position = expected_target.shift(1).fillna(0).astype(int)

assert result["target_position"].equals(expected_target)
assert result["position"].equals(expected_position)
assert result.loc[:13, "position"].eq(0).all()
assert -1 <= metrics["max_drawdown"] <= 0
assert metrics["buy_count"] >= metrics["sell_count"]
assert (result["transaction_cost"] >= 0).all()

print("检查通过：信号规则、次日仓位、最大回撤范围和交易成本均符合预期。")

检查通过：信号规则、次日仓位、最大回撤范围和交易成本均符合预期。


## Next Steps

1. 将 `stock = UNIVERSE[0]` 改成其他股票，观察行业差异。
2. 修改 `PARAMETER_PAIRS`，但不要只根据全样本最高收益挑参数。
3. 优先比较 2024 年以后的样本外结果，并同时查看回撤、交易次数和买入持有基准。
4. 进一步研究可加入滚动样本外检验、波动率过滤和更真实的成交约束。